# Network loss case — trace analysis

Report: `Results/hotel-res-enforced-supervisor/2026-01-07_15-06-33_I---Supervisor-Enforce-Certainty-P3-B7---Hotel-reservation---Network-loss-(hotel-res-enforced-supervisor).json`

Ground truth: 99% packet loss injected on service `user` (process stays alive, only the network degrades).

The triage agent's own `symptoms` output claims *"no error traces available to further diagnose"*, but the same report's `problematic_traces` field actually contains 20 error traces on `frontend`. This notebook loads the raw report and inspects those traces directly — the evidence the pipeline collected but never showed to the LLM (see `triage_agent.py:101-115`: once a problematic pod is found, the error-trace block is replaced with a placeholder string instead of the real data).

In [ ]:
import json
import os
import pandas as pd

root_dir = os.path.abspath(os.path.join(os.getcwd(), '../..'))

REPORT_PATH = os.path.join(
    root_dir,
    "Results/hotel-res-enforced-supervisor",
    "2026-01-07_15-06-33_I---Supervisor-Enforce-Certainty-P3-B7---Hotel-reservation---Network-loss-(hotel-res-enforced-supervisor).json",
)

with open(REPORT_PATH) as f:
    report = json.load(f)

list(report.keys())

## What the triage stage collected

In [ ]:
print("problematic_pods:")
print(json.dumps(report["problematic_pods"], indent=2))

In [ ]:
print("symptoms actually passed to the RCA planner:")
print(json.dumps(report["symptoms"], indent=2))

Note the evidence string: *"no error traces available to further diagnose"*. The next cell shows this is false — the traces were there, just discarded before reaching the LLM.

In [ ]:
traces = report["problematic_traces"]["traces"]
df = pd.DataFrame(traces)
print(f"{len(df)} error traces collected for service '{report['problematic_traces']['service']}'")
df

## Quick analysis

In [ ]:
print("Unique error messages:")
print(df["error_message"].value_counts())
print()
print("Unique service sequences (i.e. how far the request got before failing):")
print(df["sequence"].value_counts())
print()
print("Latency (ms) stats:")
print(df["latency_ms"].describe())

Two things stand out:

1. **`sequence` is always just `"frontend"`** — the trace never even reaches a second span/service. The call fails at the client-side gRPC resolver on `frontend` before it dispatches anything downstream to `user`.
2. **Latency is sub-millisecond** (max ~0.16ms), not a multi-second timeout. This is a *fail-fast* pattern, not a hang. It's consistent with the gRPC client's address resolver having an empty/stale endpoint list (a plausible side effect of degraded network connectivity to service discovery), producing `Unavailable: there is no address available` instantly — rather than a slow RPC that eventually times out.

This pattern (instant client-side failure, same message, same truncated sequence, repeated ~20x in the lookback window) is itself a distinctive fingerprint that a `frontend -> user` call path is broken at the connectivity/discovery layer — quite different from "container crashed at startup", which is what the agent ultimately concluded.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(df["latency_ms"], bins=15)
ax.set_xlabel("latency (ms)")
ax.set_ylabel("count")
ax.set_title("Latency of the 20 error traces on 'frontend'")
plt.show()

## What the RCA agent concluded instead

Built from `rca_tasks`, which were derived only from `symptoms` (the pod symptom above) — the RCA agent never had a channel back to `problematic_traces`, so it never saw any of this.

In [ ]:
print(report["final_report"]["root_cause"])
print()
print("evaluation:", json.dumps(report["evaluation"], indent=2))

## Raw Jaeger data — straight from the code path

This reuses the *live* testbed we stood up earlier (kind cluster + Hotel Reservation + the `network_loss_hotel_res-detection-1` fault, `user` service at 99% packet loss). We call the exact same class the project uses (`JaegerAPI` in `MCP-server/api/jaeger_api.py`), but stop **before** `process_trace()` runs, so what you see here is the literal `response.json()["data"]` returned by the Jaeger Query API (`jaeger_api.py:34-38`, inside `get_jaeger_traces()`) — the actual raw input the rest of the pipeline works from.

In [ ]:
import sys
from dotenv import load_dotenv

load_dotenv(os.path.join(root_dir, ".env"))
mcp_server_path = os.path.join(root_dir, "MCP-server")
if mcp_server_path not in sys.path:
    sys.path.insert(0, mcp_server_path)

from api.jaeger_api import JaegerAPI

jaeger_api = JaegerAPI()

# jaeger_api.jaeger_api.py:18 -- get_jaeger_traces() itself, no compaction applied yet
raw_traces = jaeger_api.get_jaeger_traces(service="frontend", only_errors=True, lookback="15m")

print(f"{len(raw_traces) if raw_traces else 0} raw traces returned")


This is the exact list handed back by `requests.get(f"{jaeger_url}/api/traces", params=...)` — nothing has been touched yet. Compare it with the compact version below, produced by `process_trace()` (`jaeger_api.py:46-110`), which is what actually reaches the LLM through the `get_traces` MCP tool.

In [ ]:
raw_trace = raw_traces[0]
print(json.dumps(raw_trace, indent=2))

In [ ]:
compact = jaeger_api.process_trace(raw_trace)
print(json.dumps(compact, indent=2))

## Compare span count / fields discarded

A quick tally of what `process_trace()` keeps vs. throws away for each raw trace, so the compression ratio is explicit rather than just asserted.

In [ ]:
rows = []
for t in raw_traces:
    n_spans = len(t["spans"])
    n_tags = sum(len(s.get("tags", [])) for s in t["spans"])
    n_logs = sum(len(s.get("logs", [])) for s in t["spans"])
    compact_t = jaeger_api.process_trace(t)
    rows.append({
        "traceID": t["traceID"],
        "raw_spans": n_spans,
        "raw_tags": n_tags,
        "raw_span_logs": n_logs,
        "compact_fields": len(compact_t) if compact_t else 0,
    })

pd.DataFrame(rows)

## Single-trace raw fetch (what the `get_trace` MCP tool actually returns)

Unlike `get_traces` (compact), the `get_trace(trace_id)` MCP tool (`mcp_server.py:241-253`) calls `JaegerAPI.get_trace()` (`jaeger_api.py:143-163`), which hits `/api/traces/{trace_id}` directly and returns the trace **unprocessed** — same raw shape as above. The LLM only ever sees this if it already has a `traceID` (from `get_traces`) and explicitly chooses to call it.

In [ ]:
same_trace_via_get_trace = jaeger_api.get_trace(raw_trace["traceID"])
same_trace_via_get_trace == raw_trace